# Inserting ISS Data To MySQL with SQLAlchemy

## Collecting data from the API

In [1]:
import requests
url =  "http://api.open-notify.org/iss-now.json"
response = requests.get(url)
json_response = response.json()
json_response

{'message': 'success',
 'timestamp': 1676303629,
 'iss_position': {'longitude': '-29.4059', 'latitude': '43.1305'}}

In [2]:
!pip install requests
!pip install pandas
!pip install sqlalchemy
!pip install pymysql

Extract only the pieces of info we need and store them in a dictionary

In [3]:
data_to_insert = {"latitude":[json_response["iss_position"]["latitude"]],
                  "longitude":[json_response["iss_position"]["longitude"]],
                  "iss_timestamp":[json_response["timestamp"]]}
data_to_insert

{'latitude': ['22.5814'],
 'longitude': ['20.1134'],
 'iss_timestamp': [1675958714]}

In [4]:
import pandas as pd

iss_df = pd.DataFrame.from_dict(data_to_insert)
iss_df

,latitude,longitude,iss_timestamp
0,22.5814,20.1134,1675958714


Timestamp from UNIX to proper datetime format

In [5]:
iss_df.iss_timestamp = pd.to_datetime(iss_df.iss_timestamp,unit='s')
iss_df

,latitude,longitude,iss_timestamp
0,22.5814,20.1134,2023-02-09 16:05:14


In [6]:
#!pip install pymysql

In [6]:
#!pip install sqlalchemy 
import sqlalchemy # install if needed

Specify MySQL connection.

Before this part you should have already created a schema (database) `sql_workshop` on you local mySql server with coresponding tables `iss_logs` and `iss_status`.

Data that we want to insert into a ddatabase should follow the same structure as a table in a database.

In [7]:
schema="sql_workshop"   # name of the database you want to use here
host="127.0.0.1"        # to connect to your local server
user="root"
password="asd7ab8BG766B6LOhygs" # your password!!!!
port=3306
con = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

Use pandas method `to_sql` with the argument `if_exists=append` to create the table (only the first time we run it) and insert the new rows into it.

In [8]:
iss_df.to_sql('iss_logs',         # 'iss_logs'-> table name;
              if_exists='append', # if_exists -> will create new table if doesn't exist, otherwise, 'append' - will append data to existing table;
              con=con,            # con-> connection string;
              index=False)        # index = False -> will not send index column to database
                                                               

1

Check on MySQLWorkbench that a new table `iss_position` exists within the `iss_workshop` database, and that a new row has been inserted on it. If you run the whole notebook again, another row should appear there.

We can now populate the other table `iss_status` by giving a fiction status active to the same timestamp.

In [9]:
data_to_insert = {"iss_timestamp":[json_response["timestamp"]],
                  "activity_status":["active"]}
status_df = pd.DataFrame.from_dict(data_to_insert)
status_df.iss_timestamp = pd.to_datetime(status_df.iss_timestamp,unit='s')

status_df.to_sql('iss_status',con=con,if_exists='append',index=False)

1